In [1]:
!pip uninstall -yq jax jaxlib jax-cuda12-plugin jax-cuda12-pjrt tensorflow-probability
# heads out since jax might drop support for cuda 12, currently (as of Aug 10, 2026), JAX has issues with CUDA 13
# see this post: https://github.com/jax-ml/jax/issues/37923
!pip install -Uq "jax[cuda12]" tfp-nightly blackjax inference_gym optax

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 124.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 136.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 390.9/390.9 kB 39.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 125.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 141.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.2/188.2 MB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 MB 28.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires tensorflow-probability>=0.13.0, which is not installed.


### **Package Import and Other Setups**

In [1]:
# GPU set up to accelerate performance
import os
# in case jax eats up my GPU RAM
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ['XLA_FLAGS'] = (
    '--xla_gpu_triton_gemm_any=True '
    '--xla_gpu_enable_latency_hiding_scheduler=true '
)

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import jax
import jax.numpy as jnp
from jax import random, jit, vmap, lax
import tensorflow_probability.substrates.jax as tfp
tfd = tfp.distributions
import inference_gym.using_jax as gym
import jaxlib
import blackjax

import optax

# import arviz as az
# import arviz_stats as avs

import warnings
warnings.filterwarnings('ignore')

import psutil

import gc

from google.colab import drive
from matplotlib.lines import Line2D

process = psutil.Process(os.getpid())

def mem(msg):
    print(f"{msg}: {process.memory_info().rss / 1024**2:.1f} MB")

# verification to make sure this is on a GPU
print(jax.devices())
print(jax.default_backend())

[CudaDevice(id=0)]
gpu


In [3]:
drive.mount('/content/drive')
utility_link = '/content/drive/MyDrive/JHU Stuff/Capstone/Utility_Functions/BJX_Utility.py'
with open(utility_link) as f: exec(f.read())

Mounted at /content/drive


### **Hyperparam Setups**

In [4]:
max_warmup = 1000
warmup_window = 100

window_array = np.append(np.repeat(10, 10),
                      np.repeat(warmup_window, max_warmup // warmup_window - 1))

warmup_length = np.repeat(10, len(window_array))
for i in range(len(warmup_length) - 1):
    warmup_length[i + 1] = warmup_length[i] + window_array[i + 1]

# Transition kernel for short regime
repitition = 10
num_chains_short = 2048
num_super_chains = 16

In [5]:
# quantiles for chi squared with df = 1
chi_up = 3.841459 # 95th quantile for chi squared with df = 1
chi_lo = 0.00393214  # 05th quantile for chi squared with df = 1
tau = 1e-4
M = num_chains_short // num_super_chains
nRhat_lower = np.sqrt(1 + 1 / M + tau)
eps_lower = nRhat_lower - 1
bound = [chi_lo / num_chains_short, chi_up / num_chains_short]
threshold = eps_lower

In [6]:
def trivariate_MVN_sampler(var1, var2, var3,rho,
                           num_dim, repitition,
                           warmup_length, num_chains_short,
                           num_super_chains, save_dir):
  # mean array
  mu = jnp.full(num_dim, 0.0)
  # rho setup
  rho12 = rho
  rho13 = rho
  rho23 = rho**2

  # covariance matrix
  cov = jnp.array([
    [var1, rho12*jnp.sqrt(var1*var2),
            rho13*jnp.sqrt(var1*var3)],
    [rho12*jnp.sqrt(var1*var2), var2,
            rho23*jnp.sqrt(var2*var3)],
    [rho13*jnp.sqrt(var1*var3),
     rho23*jnp.sqrt(var2*var3), var3]
  ])

  target = tfd.MultivariateNormalFullCovariance(
      loc=mu,
      covariance_matrix=cov
  )

  init_step_size = 0.5

  def target_log_prob_fn(x):
    return target.log_prob(x)

  def initialize(shape, key):
    offset = 2
    return 10*random.normal(key, shape + (num_dim,)) + offset

  mean_benchmark = target.mean()
  var_benchmark = target.variance()

  # simulation part
  Aniso_MSE_c_list = []
  Aniso_MSE_n_list = []
  Aniso_RHat_c_list = []
  Aniso_RHat_n_list = []

  base_key = random.PRNGKey(0)
  keys = random.split(base_key, repitition)

  for length in warmup_length:
    mem("Simulation Start")

    simulation(
        length, num_chains_short, num_super_chains,
        False, initialize, keys, target_log_prob_fn,
        init_step_size, repitition,
        Aniso_RHat_c_list, Aniso_MSE_c_list,
        mean_benchmark, var_benchmark
    )

    simulation(
        length, num_chains_short, num_super_chains,
        True, initialize, keys, target_log_prob_fn,
        init_step_size, repitition,
        Aniso_RHat_n_list, Aniso_MSE_n_list,
        mean_benchmark, var_benchmark
    )

  # save to dataframe and files
  MSE_c_df = pd.DataFrame(Aniso_MSE_c_list)
  R_Hat_c_df = pd.DataFrame(Aniso_RHat_c_list)
  MSE_n_df = pd.DataFrame(Aniso_MSE_n_list)
  R_Hat_n_df = pd.DataFrame(Aniso_RHat_n_list)

  # descriptive parameter label
  label = (
      f"var1_{var1:g}_var2_{var2:g}_var3_{var3:g}"
      f"_rho_{rho:g}"
  )

  MSE_c_df.to_pickle(f"{save_dir}/{label}_MSE_c.pkl")
  MSE_n_df.to_pickle(f"{save_dir}/{label}_MSE_n.pkl")
  R_Hat_c_df.to_pickle(f"{save_dir}/{label}_Rhat_c.pkl")
  R_Hat_n_df.to_pickle(f"{save_dir}/{label}_Rhat_n.pkl")

  del MSE_c_df, R_Hat_c_df, MSE_n_df, R_Hat_n_df
  gc.collect()

  print("Simulation Done, Data Saved!")


### **Experiment Part**

In [ ]:
# Set up File Directory
save_dir = "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/GeometryExperiment/Trivariate"

#### **Distribution with covariance matrix**:
$$
\begin{bmatrix}
  10 & 10\rho & \rho^2\sqrt{10} \\
  10\rho & 10 & \rho\sqrt{10} \\
  \rho^2\sqrt{10}& \rho\sqrt{10} & 1 \\
\end{bmatrix}
$$

In [ ]:
var1 = 10.0
var2 = 10.0
var3 = 1.0
num_dim = 3

In [ ]:
rho = 0.0
trivariate_MVN_sampler(var1, var2, var3, rho, num_dim, repitition,
                       warmup_length, num_chains_short, num_super_chains,save_dir)

Simulation Start: 1516.6 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.007165284361690283
Naive initialization. Warmup Length: 10; mean of MSE is: 0.003230074420571327
Simulation Start: 1898.6 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.00023335427977144718
Naive initialization. Warmup Length: 20; mean of MSE is: 0.00022359091963153332
Simulation Start: 1913.0 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.00031001243041828275
Naive initialization. Warmup Length: 30; mean of MSE is: 0.000399771990487352
Simulation Start: 1919.4 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.0006978745805099607
Naive initialization. Warmup Length: 40; mean of MSE is: 0.0005593349342234433
Simulation Start: 1923.0 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.00037316803354769945
Naive initialization. Warmup Length: 50; mean of MSE is: 0.00033035542583093047
Simulation Start: 1925.4 MB
Constraine

In [ ]:
rho = 0.3
trivariate_MVN_sampler(var1, var2, var3, rho, num_dim, repitition,
                       warmup_length, num_chains_short, num_super_chains,save_dir)

Simulation Start: 1946.1 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.012717870995402336
Naive initialization. Warmup Length: 10; mean of MSE is: 0.00732817267999053
Simulation Start: 1947.3 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.0005563489976339042
Naive initialization. Warmup Length: 20; mean of MSE is: 0.0003061051247641444
Simulation Start: 1947.6 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.0003997255698777735
Naive initialization. Warmup Length: 30; mean of MSE is: 0.0003311759210191667
Simulation Start: 1947.7 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.0005128246266394854
Naive initialization. Warmup Length: 40; mean of MSE is: 0.0005037948722019792
Simulation Start: 1947.7 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.00033677395549602807
Naive initialization. Warmup Length: 50; mean of MSE is: 0.00040155850001610816
Simulation Start: 1947.8 MB
Constrained i

In [ ]:
rho = 0.6
trivariate_MVN_sampler(var1, var2, var3, rho, num_dim, repitition,
                       warmup_length, num_chains_short, num_super_chains,save_dir)

Simulation Start: 1951.3 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.028774568811058998
Naive initialization. Warmup Length: 10; mean of MSE is: 0.019059354439377785
Simulation Start: 1954.7 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.002432803390547633
Naive initialization. Warmup Length: 20; mean of MSE is: 0.0010825787903741002
Simulation Start: 1954.7 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.0004906946560367942
Naive initialization. Warmup Length: 30; mean of MSE is: 0.0003738355590030551
Simulation Start: 1954.8 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.0005121038993820548
Naive initialization. Warmup Length: 40; mean of MSE is: 0.0005184608744457364
Simulation Start: 1954.9 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.0004262434667907655
Naive initialization. Warmup Length: 50; mean of MSE is: 0.000423153571318835
Simulation Start: 1954.9 MB
Constrained init

In [ ]:
rho = 0.9
trivariate_MVN_sampler(var1, var2, var3, rho, num_dim, repitition,
                       warmup_length, num_chains_short, num_super_chains,save_dir)

Simulation Start: 1955.6 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.1809912472963333
Naive initialization. Warmup Length: 10; mean of MSE is: 0.1117338165640831
Simulation Start: 1955.9 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.027253467589616776
Naive initialization. Warmup Length: 20; mean of MSE is: 0.01688154973089695
Simulation Start: 1955.9 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.002318920101970434
Naive initialization. Warmup Length: 30; mean of MSE is: 0.001374519313685596
Simulation Start: 1955.9 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.0008457961375825107
Naive initialization. Warmup Length: 40; mean of MSE is: 0.00086114153964445
Simulation Start: 1956.0 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.0005822450621053576
Naive initialization. Warmup Length: 50; mean of MSE is: 0.0005100102280266583
Simulation Start: 1956.0 MB
Constrained initializatio

In [ ]:
rho = -0.3
trivariate_MVN_sampler(var1, var2, var3, rho, num_dim, repitition,
                       warmup_length, num_chains_short, num_super_chains,save_dir)

Simulation Start: 1956.9 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.0047400398179888725
Naive initialization. Warmup Length: 10; mean of MSE is: 0.0009662465308792889
Simulation Start: 1958.9 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.00019207331934012473
Naive initialization. Warmup Length: 20; mean of MSE is: 0.00020065772696398199
Simulation Start: 1959.0 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.00042888210737146437
Naive initialization. Warmup Length: 30; mean of MSE is: 0.00041690460056997836
Simulation Start: 1959.0 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.0007674655644223094
Naive initialization. Warmup Length: 40; mean of MSE is: 0.0006566636729985476
Simulation Start: 1959.0 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.00044911689474247396
Naive initialization. Warmup Length: 50; mean of MSE is: 0.00040047368383966386
Simulation Start: 1959.0 MB
Constr

In [ ]:
rho = -0.6
trivariate_MVN_sampler(var1, var2, var3, rho, num_dim, repitition,
                       warmup_length, num_chains_short, num_super_chains,save_dir)

Simulation Start: 1960.0 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.010599279776215553
Naive initialization. Warmup Length: 10; mean of MSE is: 0.0005136210820637643
Simulation Start: 1960.0 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.0011063844431191683
Naive initialization. Warmup Length: 20; mean of MSE is: 0.00029171170899644494
Simulation Start: 1960.1 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.0004143243713770062
Naive initialization. Warmup Length: 30; mean of MSE is: 0.0005220257444307208
Simulation Start: 1960.1 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.0011034603230655193
Naive initialization. Warmup Length: 40; mean of MSE is: 0.0006657133344560862
Simulation Start: 1960.3 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.000567793264053762
Naive initialization. Warmup Length: 50; mean of MSE is: 0.0004048157425131649
Simulation Start: 1960.3 MB
Constrained i

In [ ]:
rho = -0.9
trivariate_MVN_sampler(var1, var2, var3, rho, num_dim, repitition,
                       warmup_length, num_chains_short, num_super_chains,save_dir)

Simulation Start: 1960.9 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.07910974323749542
Naive initialization. Warmup Length: 10; mean of MSE is: 0.0010581838432699442
Simulation Start: 1961.9 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.012465489096939564
Naive initialization. Warmup Length: 20; mean of MSE is: 0.0004388621018733829
Simulation Start: 1961.9 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.000655702780932188
Naive initialization. Warmup Length: 30; mean of MSE is: 0.0004376145079731941
Simulation Start: 1961.9 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.0005239553865976632
Naive initialization. Warmup Length: 40; mean of MSE is: 0.0007436270825564861
Simulation Start: 1961.9 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.0003753860364668071
Naive initialization. Warmup Length: 50; mean of MSE is: 0.0005405718693509698
Simulation Start: 1961.9 MB
Constrained init

#### **Distribution with covariance matrix**:
$$
\begin{bmatrix}
  100 & 100\rho & 10\rho^2 \\
  100\rho & 100 & 10\rho \\
  10\rho^2& 10\rho & 1 \\
\end{bmatrix}
$$

In [ ]:
var1 = 100.0
var2 = 100.0
var3 = 1.0
num_dim = 3

In [ ]:
rho = 0.0
trivariate_MVN_sampler(var1, var2, var3, rho, num_dim, repitition,
                       warmup_length, num_chains_short, num_super_chains,save_dir)

Simulation Start: 1962.9 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.04347027465701103
Naive initialization. Warmup Length: 10; mean of MSE is: 0.018959160894155502
Simulation Start: 1963.2 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.025955915451049805
Naive initialization. Warmup Length: 20; mean of MSE is: 0.011482501402497292
Simulation Start: 1963.2 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.014406440779566765
Naive initialization. Warmup Length: 30; mean of MSE is: 0.005793428979814053
Simulation Start: 1963.2 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.00937740784138441
Naive initialization. Warmup Length: 40; mean of MSE is: 0.004249310586601496
Simulation Start: 1963.2 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.005062813404947519
Naive initialization. Warmup Length: 50; mean of MSE is: 0.002295197220519185
Simulation Start: 1963.2 MB
Constrained initializati

In [ ]:
rho = 0.3
trivariate_MVN_sampler(var1, var2, var3, rho, num_dim, repitition,
                       warmup_length, num_chains_short, num_super_chains,save_dir)

Simulation Start: 1963.9 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.045283131301403046
Naive initialization. Warmup Length: 10; mean of MSE is: 0.0233002956956625
Simulation Start: 1964.2 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.027413848787546158
Naive initialization. Warmup Length: 20; mean of MSE is: 0.016108151525259018
Simulation Start: 1964.2 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.015554015524685383
Naive initialization. Warmup Length: 30; mean of MSE is: 0.00969624426215887
Simulation Start: 1964.2 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.011361891403794289
Naive initialization. Warmup Length: 40; mean of MSE is: 0.0074483612552285194
Simulation Start: 1964.2 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.0070076570846140385
Naive initialization. Warmup Length: 50; mean of MSE is: 0.004545651841908693
Simulation Start: 1964.2 MB
Constrained initializat

In [ ]:
rho = 0.6
trivariate_MVN_sampler(var1, var2, var3, rho, num_dim, repitition,
                       warmup_length, num_chains_short, num_super_chains,save_dir)

Simulation Start: 1964.6 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.046290379017591476
Naive initialization. Warmup Length: 10; mean of MSE is: 0.028445830568671227
Simulation Start: 1964.6 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.03232448920607567
Naive initialization. Warmup Length: 20; mean of MSE is: 0.02105880156159401
Simulation Start: 1964.6 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.020968258380889893
Naive initialization. Warmup Length: 30; mean of MSE is: 0.014022293500602245
Simulation Start: 1964.7 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.01699899509549141
Naive initialization. Warmup Length: 40; mean of MSE is: 0.010697045363485813
Simulation Start: 1964.7 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.011943237856030464
Naive initialization. Warmup Length: 50; mean of MSE is: 0.007593946997076273
Simulation Start: 1964.7 MB
Constrained initializatio

In [ ]:
rho = 0.9
trivariate_MVN_sampler(var1, var2, var3, rho, num_dim, repitition,
                       warmup_length, num_chains_short, num_super_chains,save_dir)

Simulation Start: 1965.1 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.05765689164400101
Naive initialization. Warmup Length: 10; mean of MSE is: 0.03880404680967331
Simulation Start: 1965.1 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.045933183282613754
Naive initialization. Warmup Length: 20; mean of MSE is: 0.03172791749238968
Simulation Start: 1965.1 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.03575880452990532
Naive initialization. Warmup Length: 30; mean of MSE is: 0.023384135216474533
Simulation Start: 1965.1 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.03159302473068237
Naive initialization. Warmup Length: 40; mean of MSE is: 0.020167328417301178
Simulation Start: 1965.1 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.025230249390006065
Naive initialization. Warmup Length: 50; mean of MSE is: 0.015175418928265572
Simulation Start: 1965.1 MB
Constrained initialization.

In [ ]:
rho = -0.3
trivariate_MVN_sampler(var1, var2, var3, rho, num_dim, repitition,
                       warmup_length, num_chains_short, num_super_chains,save_dir)

Simulation Start: 1965.5 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.04025019705295563
Naive initialization. Warmup Length: 10; mean of MSE is: 0.015407470054924488
Simulation Start: 1965.6 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.024074425920844078
Naive initialization. Warmup Length: 20; mean of MSE is: 0.00804058462381363
Simulation Start: 1965.6 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.01272634882479906
Naive initialization. Warmup Length: 30; mean of MSE is: 0.003194144694134593
Simulation Start: 1965.6 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.008433002047240734
Naive initialization. Warmup Length: 40; mean of MSE is: 0.002321552950888872
Simulation Start: 1965.6 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.004293411038815975
Naive initialization. Warmup Length: 50; mean of MSE is: 0.0012620544293895364
Simulation Start: 1965.6 MB
Constrained initializati

In [ ]:
rho = -0.6
trivariate_MVN_sampler(var1, var2, var3, rho, num_dim, repitition,
                       warmup_length, num_chains_short, num_super_chains,save_dir)

Simulation Start: 1965.8 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.03406209498643875
Naive initialization. Warmup Length: 10; mean of MSE is: 0.01086385827511549
Simulation Start: 1965.8 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.0198637954890728
Naive initialization. Warmup Length: 20; mean of MSE is: 0.003968501929193735
Simulation Start: 1965.8 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.01128259114921093
Naive initialization. Warmup Length: 30; mean of MSE is: 0.001316399546340108
Simulation Start: 1965.8 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.00762363662943244
Naive initialization. Warmup Length: 40; mean of MSE is: 0.001091931015253067
Simulation Start: 1965.8 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.005505718756467104
Naive initialization. Warmup Length: 50; mean of MSE is: 0.000800283276475966
Simulation Start: 1965.8 MB
Constrained initialization. 

In [ ]:
rho = -0.9
trivariate_MVN_sampler(var1, var2, var3, rho, num_dim, repitition,
                       warmup_length, num_chains_short, num_super_chains,save_dir)

Simulation Start: 1966.1 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.028401201590895653
Naive initialization. Warmup Length: 10; mean of MSE is: 0.0017968574538826942
Simulation Start: 1966.4 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.023616988211870193
Naive initialization. Warmup Length: 20; mean of MSE is: 0.00040959156467579305
Simulation Start: 1966.4 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.01862979307770729
Naive initialization. Warmup Length: 30; mean of MSE is: 0.0005981124704703689
Simulation Start: 1966.4 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.015414093621075153
Naive initialization. Warmup Length: 40; mean of MSE is: 0.0007613993366248906
Simulation Start: 1966.4 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.011465380899608135
Naive initialization. Warmup Length: 50; mean of MSE is: 0.0006486610509455204
Simulation Start: 1966.4 MB
Constrained initi

#### **Distribution with covariance matrix**:
$$
\begin{bmatrix}
  100 & 100\rho & 10\rho^2 \\
  100\rho & 10 & 10\rho \\
  10\rho^2& 10\rho & 1 \\
\end{bmatrix}
$$

In [1]:
var1 = 100.0
var2 = 10.0
var3 = 1.0
num_dim = 3

In [ ]:
rho = 0.0
trivariate_MVN_sampler(var1, var2, var3, rho, num_dim, repitition,
                       warmup_length, num_chains_short, num_super_chains,save_dir)

Simulation Start: 1966.6 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.013854135759174824
Naive initialization. Warmup Length: 10; mean of MSE is: 0.011382153257727623
Simulation Start: 1966.6 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.007474807091057301
Naive initialization. Warmup Length: 20; mean of MSE is: 0.006459626369178295
Simulation Start: 1966.6 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.00425211526453495
Naive initialization. Warmup Length: 30; mean of MSE is: 0.003896885784342885
Simulation Start: 1966.6 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.003088383236899972
Naive initialization. Warmup Length: 40; mean of MSE is: 0.002736742375418544
Simulation Start: 1966.6 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.0020388872362673283
Naive initialization. Warmup Length: 50; mean of MSE is: 0.0016441087936982512
Simulation Start: 1966.6 MB
Constrained initializ

In [ ]:
rho = 0.3
trivariate_MVN_sampler(var1, var2, var3, rho, num_dim, repitition,
                       warmup_length, num_chains_short, num_super_chains,save_dir)

Simulation Start: 1967.0 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.022091886028647423
Naive initialization. Warmup Length: 10; mean of MSE is: 0.017085852101445198
Simulation Start: 1967.0 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.011147375218570232
Naive initialization. Warmup Length: 20; mean of MSE is: 0.009442201815545559
Simulation Start: 1967.0 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.005750873126089573
Naive initialization. Warmup Length: 30; mean of MSE is: 0.005110905505716801
Simulation Start: 1967.0 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.004163373727351427
Naive initialization. Warmup Length: 40; mean of MSE is: 0.003926202189177275
Simulation Start: 1967.0 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.0026094121858477592
Naive initialization. Warmup Length: 50; mean of MSE is: 0.0022788704372942448
Simulation Start: 1967.0 MB
Constrained initiali

In [ ]:
rho = 0.6
trivariate_MVN_sampler(var1, var2, var3, rho, num_dim, repitition,
                       warmup_length, num_chains_short, num_super_chains,save_dir)

Simulation Start: 1967.1 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.029375845566391945
Naive initialization. Warmup Length: 10; mean of MSE is: 0.026543831452727318
Simulation Start: 1967.1 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.0180569626390934
Naive initialization. Warmup Length: 20; mean of MSE is: 0.016863156110048294
Simulation Start: 1967.1 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.010417087003588676
Naive initialization. Warmup Length: 30; mean of MSE is: 0.010131431743502617
Simulation Start: 1967.1 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.007204397115856409
Naive initialization. Warmup Length: 40; mean of MSE is: 0.007384619675576687
Simulation Start: 1967.1 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.004636308643966913
Naive initialization. Warmup Length: 50; mean of MSE is: 0.0048398240469396114
Simulation Start: 1967.1 MB
Constrained initializat

In [ ]:
rho = 0.9
trivariate_MVN_sampler(var1, var2, var3, rho, num_dim, repitition,
                       warmup_length, num_chains_short, num_super_chains,save_dir)

Simulation Start: 1967.8 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.057765256613492966
Naive initialization. Warmup Length: 10; mean of MSE is: 0.05004525184631348
Simulation Start: 1967.8 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.041204266250133514
Naive initialization. Warmup Length: 20; mean of MSE is: 0.03454604372382164
Simulation Start: 1967.9 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.027379145845770836
Naive initialization. Warmup Length: 30; mean of MSE is: 0.02304571494460106
Simulation Start: 1968.0 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.02115202508866787
Naive initialization. Warmup Length: 40; mean of MSE is: 0.018099334090948105
Simulation Start: 1968.0 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.014396106824278831
Naive initialization. Warmup Length: 50; mean of MSE is: 0.01188261155039072
Simulation Start: 1968.1 MB
Constrained initialization.

In [ ]:
rho = -0.3
trivariate_MVN_sampler(var1, var2, var3, rho, num_dim, repitition,
                       warmup_length, num_chains_short, num_super_chains,save_dir)

Simulation Start: 1968.4 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.013811208307743073
Naive initialization. Warmup Length: 10; mean of MSE is: 0.009699935093522072
Simulation Start: 1968.5 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.007756256498396397
Naive initialization. Warmup Length: 20; mean of MSE is: 0.005530643276870251
Simulation Start: 1968.5 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.004585319198668003
Naive initialization. Warmup Length: 30; mean of MSE is: 0.003680035937577486
Simulation Start: 1968.5 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.0031767506152391434
Naive initialization. Warmup Length: 40; mean of MSE is: 0.002607126720249653
Simulation Start: 1968.5 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.0018862651195377111
Naive initialization. Warmup Length: 50; mean of MSE is: 0.0018370996695011854
Simulation Start: 1968.5 MB
Constrained initial

In [ ]:
rho = -0.6
trivariate_MVN_sampler(var1, var2, var3, rho, num_dim, repitition,
                       warmup_length, num_chains_short, num_super_chains,save_dir)

Simulation Start: 1968.9 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.017027297988533974
Naive initialization. Warmup Length: 10; mean of MSE is: 0.010325498878955841
Simulation Start: 1969.6 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.011359035968780518
Naive initialization. Warmup Length: 20; mean of MSE is: 0.006823144853115082
Simulation Start: 1969.6 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.006278345827013254
Naive initialization. Warmup Length: 30; mean of MSE is: 0.004410178400576115
Simulation Start: 1969.6 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.0041304221376776695
Naive initialization. Warmup Length: 40; mean of MSE is: 0.003810222027823329
Simulation Start: 1969.6 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.0027861064299941063
Naive initialization. Warmup Length: 50; mean of MSE is: 0.002165602520108223
Simulation Start: 1969.6 MB
Constrained initiali

In [ ]:
rho = -0.9
trivariate_MVN_sampler(var1, var2, var3, rho, num_dim, repitition,
                       warmup_length, num_chains_short, num_super_chains,save_dir)

Simulation Start: 1969.8 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.02604769729077816
Naive initialization. Warmup Length: 10; mean of MSE is: 0.012252528220415115
Simulation Start: 1969.8 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.019067415967583656
Naive initialization. Warmup Length: 20; mean of MSE is: 0.008081837557256222
Simulation Start: 1969.8 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.011735095642507076
Naive initialization. Warmup Length: 30; mean of MSE is: 0.005742102395743132
Simulation Start: 1969.8 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.008823933079838753
Naive initialization. Warmup Length: 40; mean of MSE is: 0.005094471387565136
Simulation Start: 1969.8 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.005846885032951832
Naive initialization. Warmup Length: 50; mean of MSE is: 0.0031932778656482697
Simulation Start: 1969.9 MB
Constrained initializa

#### **Distribution with covariance matrix**:
$$
\begin{bmatrix}
  1000 & 100\rho & 10\rho^2 \\
  100\rho & 100 & 10\rho \\
  10\rho^2& 10\rho & 1 \\
\end{bmatrix}
$$

In [9]:
var1 = 1000.0
var2 = 100.0
var3 = 1.0
num_dim = 3

In [9]:
rho = 0.0
trivariate_MVN_sampler(var1, var2, var3, rho, num_dim, repitition,
                       warmup_length, num_chains_short, num_super_chains,save_dir)

Simulation Start: 1517.0 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.03233847767114639
Naive initialization. Warmup Length: 10; mean of MSE is: 0.010283350013196468
Simulation Start: 1900.5 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.020962323993444443
Naive initialization. Warmup Length: 20; mean of MSE is: 0.006817986257374287
Simulation Start: 1917.2 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.011793648824095726
Naive initialization. Warmup Length: 30; mean of MSE is: 0.0038830724079161882
Simulation Start: 1921.9 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.008197320625185966
Naive initialization. Warmup Length: 40; mean of MSE is: 0.003210417926311493
Simulation Start: 1926.1 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.005223266780376434
Naive initialization. Warmup Length: 50; mean of MSE is: 0.0022234548814594746
Simulation Start: 1928.4 MB
Constrained initializ

In [10]:
rho = 0.3
trivariate_MVN_sampler(var1, var2, var3, rho, num_dim, repitition,
                       warmup_length, num_chains_short, num_super_chains,save_dir)

Simulation Start: 1949.3 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.030917635187506676
Naive initialization. Warmup Length: 10; mean of MSE is: 0.010380956344306469
Simulation Start: 1951.2 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.019729427993297577
Naive initialization. Warmup Length: 20; mean of MSE is: 0.007450055330991745
Simulation Start: 1951.3 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.011284181848168373
Naive initialization. Warmup Length: 30; mean of MSE is: 0.004388881381601095
Simulation Start: 1951.5 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.008366826921701431
Naive initialization. Warmup Length: 40; mean of MSE is: 0.003879981115460396
Simulation Start: 1951.6 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.005302182398736477
Naive initialization. Warmup Length: 50; mean of MSE is: 0.0028022266924381256
Simulation Start: 1951.7 MB
Constrained initializ

In [11]:
rho = 0.6
trivariate_MVN_sampler(var1, var2, var3, rho, num_dim, repitition,
                       warmup_length, num_chains_short, num_super_chains,save_dir)

Simulation Start: 1953.5 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.02599659189581871
Naive initialization. Warmup Length: 10; mean of MSE is: 0.01020676176995039
Simulation Start: 1956.8 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.015860823914408684
Naive initialization. Warmup Length: 20; mean of MSE is: 0.007237779442220926
Simulation Start: 1957.0 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.00859563983976841
Naive initialization. Warmup Length: 30; mean of MSE is: 0.0047479174099862576
Simulation Start: 1957.1 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.0061850110068917274
Naive initialization. Warmup Length: 40; mean of MSE is: 0.0041315918788313866
Simulation Start: 1957.1 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.004555706400424242
Naive initialization. Warmup Length: 50; mean of MSE is: 0.003460000501945615
Simulation Start: 1957.2 MB
Constrained initializa

In [12]:
rho = 0.9
trivariate_MVN_sampler(var1, var2, var3, rho, num_dim, repitition,
                       warmup_length, num_chains_short, num_super_chains,save_dir)

Simulation Start: 1958.4 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.015519203618168831
Naive initialization. Warmup Length: 10; mean of MSE is: 0.00874144397675991
Simulation Start: 1958.5 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.0081360824406147
Naive initialization. Warmup Length: 20; mean of MSE is: 0.006347929127514362
Simulation Start: 1958.6 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.005942065268754959
Naive initialization. Warmup Length: 30; mean of MSE is: 0.005428785923868418
Simulation Start: 1958.7 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.00562957813963294
Naive initialization. Warmup Length: 40; mean of MSE is: 0.005235348828136921
Simulation Start: 1958.7 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.005457667633891106
Naive initialization. Warmup Length: 50; mean of MSE is: 0.0049360208213329315
Simulation Start: 1958.8 MB
Constrained initializatio

In [13]:
rho = -0.3
trivariate_MVN_sampler(var1, var2, var3, rho, num_dim, repitition,
                       warmup_length, num_chains_short, num_super_chains,save_dir)

Simulation Start: 1959.9 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.030413052067160606
Naive initialization. Warmup Length: 10; mean of MSE is: 0.009072072803974152
Simulation Start: 1962.3 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.018917527049779892
Naive initialization. Warmup Length: 20; mean of MSE is: 0.006035306490957737
Simulation Start: 1962.4 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.010074637830257416
Naive initialization. Warmup Length: 30; mean of MSE is: 0.0029474860057234764
Simulation Start: 1962.4 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.006968361325562
Naive initialization. Warmup Length: 40; mean of MSE is: 0.002404619473963976
Simulation Start: 1962.4 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.004070615395903587
Naive initialization. Warmup Length: 50; mean of MSE is: 0.0017275121062994003
Simulation Start: 1962.6 MB
Constrained initializat

In [ ]:
rho = -0.6
trivariate_MVN_sampler(var1, var2, var3, rho, num_dim, repitition,
                       warmup_length, num_chains_short, num_super_chains,save_dir)

Simulation Start: 1514.8 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.024431202560663223
Naive initialization. Warmup Length: 10; mean of MSE is: 0.007232688367366791
Simulation Start: 1769.2 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.014087339863181114
Naive initialization. Warmup Length: 20; mean of MSE is: 0.004035957623273134
Simulation Start: 1774.5 MB


In [15]:
rho = -0.9
trivariate_MVN_sampler(var1, var2, var3, rho, num_dim, repitition,
                       warmup_length, num_chains_short, num_super_chains,save_dir)

Simulation Start: 1963.5 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.011706766672432423
Naive initialization. Warmup Length: 10; mean of MSE is: 0.0030190797988325357
Simulation Start: 1964.8 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.004735199734568596
Naive initialization. Warmup Length: 20; mean of MSE is: 0.0013708841288462281
Simulation Start: 1964.9 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.003332345513626933
Naive initialization. Warmup Length: 30; mean of MSE is: 0.001514485222287476
Simulation Start: 1964.9 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.0028787669725716114
Naive initialization. Warmup Length: 40; mean of MSE is: 0.0017504164716228843
Simulation Start: 1964.9 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.0027646676171571016
Naive initialization. Warmup Length: 50; mean of MSE is: 0.0017255035927519202
Simulation Start: 1964.9 MB
Constrained init

In [16]:
from google.colab import runtime
runtime.unassign()